In [1]:
%pip install -U scikit-learn

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [3]:
%pip install statsmodels

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns 
import numpy as np
import sklearn

In [4]:
import statsmodels.api as sm

from statsmodels.stats.outliers_influence import (
    variance_inflation_factor
)
from sklearn.model_selection import (
    TimeSeriesSplit,
    cross_validate
)

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from sklearn.linear_model import Ridge
from sklearn.ensemble import (
    RandomForestRegressor,
    GradientBoostingRegressor
)
from sklearn.svm import SVR

from sklearn.metrics import (
    mean_absolute_error,
    mean_absolute_percentage_error,
    mean_squared_error,
    r2_score
)

In [5]:
#Confifuración de visualización
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (12,6)

#Cargar datos
df = pd.read_csv('data/dataset_bogota_5years.csv')
df['Fecha'] = pd.to_datetime(df['Fecha'])

df.head()

,Fecha,ALLSKY_SFC_SW_DWN,T2M,T2M_MAX,T2M_MIN,T2M_RANGE,RH2M,CLOUD_AMT,CLRSKY_SFC_SW_DWN,PRECTOTCORR
0,2020-01-01,4.8713,18.87,23.47,15.30,8.17,85.65,75.95,6.8179,1.67
1,2020-01-02,6.0058,19.03,24.79,15.38,9.41,84.94,45.61,6.8777,0.95
2,2020-01-03,6.1850,19.09,25.37,13.59,11.78,77.95,32.61,6.9154,0.13
3,2020-01-04,6.3425,18.60,24.91,13.17,11.74,79.60,34.75,6.8580,0.00
4,2020-01-05,6.2306,18.42,23.68,14.00,9.68,86.14,47.65,6.8774,0.00


## Pregunta 4

In [6]:
df_modelo = (
    df.copy()
    .sort_values('Fecha')
    .reset_index(drop=True)
)

df_modelo['Fecha'] = pd.to_datetime(
    df_modelo['Fecha']
)

objetivo_modelo = 'ALLSKY_SFC_SW_DWN'

variables_historicas = [
    'ALLSKY_SFC_SW_DWN',
    'CLRSKY_SFC_SW_DWN',
    'CLOUD_AMT',
    'RH2M',
    'T2M',
    'T2M_RANGE'
]

# Comprobar que todas las variables existan
variables_faltantes = [
    variable
    for variable in variables_historicas
    if variable not in df_modelo.columns
]

if variables_faltantes:
    raise ValueError(
        f'Faltan estas variables: {variables_faltantes}'
    )

# Crear retrasos de 1 a 7 días
columnas_retrasos = []

for variable in variables_historicas:
    for retraso in range(1, 8):
        nombre_columna = (
            f'{variable}_lag_{retraso}'
        )

        df_modelo[nombre_columna] = (
            df_modelo[variable]
            .shift(retraso)
        )

        columnas_retrasos.append(nombre_columna)

In [7]:
columnas_resumen = []

for variable in variables_historicas:
    columna_media = f'{variable}_media_7d'
    columna_desviacion = f'{variable}_std_7d'

    serie_anterior = (
        df_modelo[variable]
        .shift(1)
    )

    df_modelo[columna_media] = (
        serie_anterior
        .rolling(window=7)
        .mean()
    )

    df_modelo[columna_desviacion] = (
        serie_anterior
        .rolling(window=7)
        .std()
    )

    columnas_resumen.extend([
        columna_media,
        columna_desviacion
    ])

In [8]:
df_modelo['Dia_del_anio'] = (
    df_modelo['Fecha'].dt.dayofyear
)

df_modelo['Dia_anio_seno'] = np.sin(
    2 * np.pi
    * df_modelo['Dia_del_anio']
    / 365.25
)

df_modelo['Dia_anio_coseno'] = np.cos(
    2 * np.pi
    * df_modelo['Dia_del_anio']
    / 365.25
)

columnas_temporales = [
    'Dia_anio_seno',
    'Dia_anio_coseno'
]

predictores_modelo = (
    columnas_retrasos
    + columnas_resumen
    + columnas_temporales
)

datos_modelo = (
    df_modelo[
        ['Fecha', objetivo_modelo]
        + predictores_modelo
    ]
    .dropna()
    .reset_index(drop=True)
)

print(
    'Observaciones disponibles:',
    len(datos_modelo)
)

print(
    'Cantidad de predictores:',
    len(predictores_modelo)
)

print(
    'Periodo:',
    datos_modelo['Fecha'].min(),
    'a',
    datos_modelo['Fecha'].max()
)

Observaciones disponibles: 1820
Cantidad de predictores: 56
Periodo: 2020-01-08 00:00:00 a 2024-12-31 00:00:00


In [9]:
entrenamiento = datos_modelo[
    datos_modelo['Fecha'].dt.year <= 2023
].copy()

prueba_2024 = datos_modelo[
    datos_modelo['Fecha'].dt.year == 2024
].copy()

X_entrenamiento = entrenamiento[
    predictores_modelo
]

y_entrenamiento = entrenamiento[
    objetivo_modelo
]

X_prueba = prueba_2024[
    predictores_modelo
]

y_prueba = prueba_2024[
    objetivo_modelo
]

print(
    'Entrenamiento:',
    entrenamiento['Fecha'].min(),
    'a',
    entrenamiento['Fecha'].max(),
    '-',
    len(entrenamiento),
    'días'
)

print(
    'Prueba:',
    prueba_2024['Fecha'].min(),
    'a',
    prueba_2024['Fecha'].max(),
    '-',
    len(prueba_2024),
    'días'
)

Entrenamiento: 2020-01-08 00:00:00 a 2023-12-31 00:00:00 - 1454 días
Prueba: 2024-01-01 00:00:00 a 2024-12-31 00:00:00 - 366 días


In [10]:
modelos = {
    'Ridge': Pipeline([
        (
            'escalador',
            StandardScaler()
        ),
        (
            'modelo',
            Ridge(alpha=1.0)
        )
    ]),

    'Random Forest': RandomForestRegressor(
        n_estimators=300,
        min_samples_leaf=2,
        random_state=42,
        n_jobs=-1
    ),

    'Gradient Boosting': GradientBoostingRegressor(
        n_estimators=150,
        learning_rate=0.05,
        max_depth=3,
        min_samples_leaf=3,
        random_state=42
    ),

    'SVR': Pipeline([
        (
            'escalador',
            StandardScaler()
        ),
        (
            'modelo',
            SVR(
                kernel='rbf',
                C=10,
                epsilon=0.1
            )
        )
    ])
}

In [11]:
validacion_temporal = TimeSeriesSplit(
    n_splits=5
)

metricas_cv = {
    'MAE': 'neg_mean_absolute_error',
    'RMSE': 'neg_root_mean_squared_error',
    'MAPE': 'neg_mean_absolute_percentage_error',
    'R2': 'r2'
}

resultados_cv = []

for nombre, modelo in modelos.items():

    resultado = cross_validate(
        estimator=modelo,
        X=X_entrenamiento,
        y=y_entrenamiento,
        cv=validacion_temporal,
        scoring=metricas_cv,
        n_jobs=-1
    )

    resultados_cv.append({
        'Modelo': nombre,
        'MAPE medio (%)': (
            -resultado['test_MAPE'].mean()
            * 100
        ),
        'Desv. MAPE': (
            resultado['test_MAPE']
            .std()
            * 100
        ),
        'MAE medio': (
            -resultado['test_MAE'].mean()
        ),
        'RMSE medio': (
            -resultado['test_RMSE'].mean()
        ),
        'R² medio': (
            resultado['test_R2'].mean()
        )
    })

tabla_validacion = (
    pd.DataFrame(resultados_cv)
    .sort_values('MAPE medio (%)')
    .reset_index(drop=True)
)

display(tabla_validacion.round(4))

,Modelo,MAPE medio (%),Desv. MAPE,MAE medio,RMSE medio,R² medio
0,Random Forest,13.8453,1.4388,0.5883,0.7588,0.0623
1,Gradient Boosting,14.2429,1.5139,0.6062,0.7819,0.0020
2,Ridge,14.6454,2.0388,0.6174,0.7996,-0.0521
3,SVR,16.2167,1.9448,0.6980,0.8802,-0.2640


In [12]:
def calcular_metricas(
    nombre,
    valores_reales,
    predicciones
):
    return {
        'Modelo': nombre,

        'MAPE (%)': (
            mean_absolute_percentage_error(
                valores_reales,
                predicciones
            )
            * 100
        ),

        'MAE': mean_absolute_error(
            valores_reales,
            predicciones
        ),

        'RMSE': np.sqrt(
            mean_squared_error(
                valores_reales,
                predicciones
            )
        ),

        'R²': r2_score(
            valores_reales,
            predicciones
        )
    }

In [13]:
resultados_2024 = []

prediccion_persistencia = prueba_2024[
    'ALLSKY_SFC_SW_DWN_lag_1'
]

resultados_2024.append(
    calcular_metricas(
        'Persistencia: día anterior',
        y_prueba,
        prediccion_persistencia
    )
)

prediccion_media_7d = prueba_2024[
    'ALLSKY_SFC_SW_DWN_media_7d'
]

resultados_2024.append(
    calcular_metricas(
        'Promedio de 7 días',
        y_prueba,
        prediccion_media_7d
    )
)

In [14]:
modelos_entrenados = {}
predicciones_2024 = {}

for nombre, modelo in modelos.items():

    modelo.fit(
        X_entrenamiento,
        y_entrenamiento
    )

    prediccion = modelo.predict(
        X_prueba
    )

    # La irradiación no puede ser negativa
    prediccion = np.clip(
        prediccion,
        0,
        None
    )

    modelos_entrenados[nombre] = modelo
    predicciones_2024[nombre] = prediccion

    resultados_2024.append(
        calcular_metricas(
            nombre,
            y_prueba,
            prediccion
        )
    )

tabla_prueba_2024 = (
    pd.DataFrame(resultados_2024)
    .sort_values('MAPE (%)')
    .reset_index(drop=True)
)

display(tabla_prueba_2024.round(4))

,Modelo,MAPE (%),MAE,RMSE,R²
0,Ridge,12.6878,0.5580,0.7191,0.3172
1,Random Forest,13.4890,0.5947,0.7661,0.2251
2,Gradient Boosting,13.5570,0.5988,0.7758,0.2053
3,Promedio de 7 días,14.8271,0.6488,0.8338,0.0820
4,SVR,15.6818,0.7084,0.8872,-0.0392
5,Persistencia: día anterior,16.9491,0.7492,0.9651,-0.2298
